# Aplicación 2 — Detección con RF-DETR (transformer) sobre las modalidades

Replica el experimento de YOLO con **RF-DETR** (detector tipo DETR de Roboflow, basado en transformadores), para contrastar un paradigma distinto al de una etapa. Mismas 4 modalidades. Colab con GPU.


## 1. Instalación


In [ ]:
!pip -q install rfdetr roboflow supervision pyyaml opencv-python-headless
import torch; print('CUDA', torch.cuda.is_available())


## 2. Dataset (Roboflow → formato COCO, requerido por RF-DETR)


In [ ]:
from roboflow import Roboflow
rf=Roboflow(api_key='TU_API_KEY')
dataset=rf.workspace('TU_WS').project('TU_PROYECTO').version(1).download('coco')  # RF-DETR usa COCO


## 3. Generar las 4 modalidades (conservando anotaciones COCO)
Aplicar la fusión a cada imagen del dataset COCO manteniendo el `annotations.json`. Reusar la función de fusión del repo; las cajas son válidas para todas las modalidades.


In [ ]:
# Reutilizá las clases de fusión del repo (igual que en el notebook de YOLO).
from src.fusion import TopHatFusion
from src.fusion.optimal_top_hat import OptimalMultiscaleFusion
import cv2, numpy as np, shutil, json as _json
from pathlib import Path
FUSERS={'VIS':lambda v,i:v,'IR':lambda v,i:i,
        'Anterior_TopHat':lambda v,i:TopHatFusion('disk',5).fuse(v,i),
        'Optimo_Multiescala':lambda v,i:OptimalMultiscaleFusion(n=6,base_radius=2.89,m=0.10).fuse(v,i)}
# Para cada split COCO: copiar _annotations.coco.json y regenerar las imágenes fusionadas por modalidad.
# (ver el patrón de build_variant del notebook 04; aquí se conserva el JSON COCO en lugar de labels YOLO).


## 4. Entrenar RF-DETR por modalidad y comparar


In [ ]:
from rfdetr import RFDETRBase; import pandas as pd
res={}
for name in ['VIS','IR','Anterior_TopHat','Optimo_Multiescala']:
    ds_dir=f'ds_{name}'   # carpeta COCO de la modalidad (train/valid/test + _annotations.coco.json)
    model=RFDETRBase()
    model.train(dataset_dir=ds_dir, epochs=30, batch_size=4, lr=1e-4, output_dir=f'out_{name}')
    # Evaluar mAP en el split de test (usar supervision.MeanAveragePrecision con las predicciones):
    # preds = [model.predict(img) for img in test_images]  -> mAP50, mAP50_95
    # res[name] = {'mAP50':..., 'mAP50_95':..., 'P':..., 'R':..., 'F1':...}
# pd.DataFrame(res).T.to_csv('rfdetr_por_modalidad.csv')


## 5. Producción y lectura
Medir latencia/FPS y tamaño del modelo (export ONNX si aplica). Comparar RF-DETR vs YOLO por modalidad para ver si la ventaja de la fusión es consistente entre un detector de una etapa y uno basado en transformadores.
